In [32]:
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import os
import json
import ROOT

In [33]:
os.environ['ttH_yy_DIR'] = '/eos/user/e/elmazzeo/ttH@FCC-hh/results/2025-03-05/'

In [42]:
basedir = os.path.join(os.environ.get('ttH_yy_DIR'), 'final')

process = {
    'ttHyy' : { 'sample_list' : ['mgp8_pp_tth01j_5f_haa'], 
               'label' : r'$ttH$ $\rightarrow$ $\gamma\gamma$ (100 TeV)', 'kfactor' : 1.},
    #'yy_jets' : { 'sample_list' : ['mgp8_pp_jjaa_5f'], 
    #           'label' : r'$\gamma\gamma$ + jets'},
#    'ttyy' : { 'sample_list' : ['mgp8_pp_ttaa_semilep_5f_100TeV'], 
#               'label' : '$tt\gamma\gamma$ (100 TeV)', 'kfactor' : 1.5},
#    'Vyy_jets' : { 'sample_list' : ['mgp8_pp_Vaajj_HF_5f_84TeV'], 
#               'label' : 'V$\gamma\gamma$+jets (84 TeV)', 'kfactor' : 1.}
}

selection = {
            "nocuts" : "All events", # all events
            "photons": "$\geq$ 2 photons",
            "photons_rel_pt": "Rel. $p_{T}$ cuts",
            "photons_myy_window": "110 < $m_{\gamma\gamma}$ < 140 GeV",
            "preselection": "$\geq$ 2 b-jets",
            "lep_channel" : "$\geq$ 1 lepton",
            #"preselection_myy_window_narrow": "120 < $m_{\gamma\gamma}$ < 130 GeV",
            #"lep_channel" : "Photon and $b$-jet sel., $\geq$ 1 lepton",
            #"ee_channel" : "$\geq$ 2 electrons",
            #"mumu_channel" : "$\geq$ 2 muons",
            #"emu_channel" : "$\geq$ 1 electron and $\geq$ 1 muon",
            #"boosted_analysis_pT_yy_100" : "$p_{T}(\gamma\gamma)$>100 GeV",
            #"boosted_analysis_pT_yy_150" : "$p_{T}(\gamma\gamma)$>150 GeV",
            #"boosted_analysis_pT_yy_200" : "$p_{T}(\gamma\gamma)$>200 GeV",
            #"boosted_analysis_pT_yy_250" : "$p_{T}(\gamma\gamma)$>250 GeV",
            #"boosted_analysis_pT_yy_300" : "$p_{T}(\gamma\gamma)$>300 GeV",
            #"pT_yy_bin1" : "0$\leq p_{T}(\gamma\gamma)$< 60 GeV",
            #"pT_yy_bin2" : "60$\leq p_{T}(\gamma\gamma)$< 120 GeV",
            #"pT_yy_bin3" : "120$\leq p_{T}(\gamma\gamma)$< 200 GeV",
            #"pT_yy_bin4" : "200$\leq p_{T}(\gamma\gamma)$< 300 GeV",
            #"pT_yy_bin5" : "$p_{T}(\gamma\gamma)\geq$ 300 GeV",
}

variables = ['weight']

process_infos = ["/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v07_II.json",
                 "/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v06_II.json"]

lumi = 3e7 # 3 * 10^7 pb-1 = 3 ab-1

In [43]:
process_dicts = []
for inputname in process_infos :
    with open(inputname, 'r') as f :
        process_dicts.append(json.load(f))

In [44]:
df = {}
sow = {}

In [45]:
for p in process.keys() :
    df[p] = {}
    print(p)
    for s in selection.keys() :
        df1 = []
        sow[p] = []
        print("\t"+s)
        for sample in process[p]['sample_list'] :
            print("\t\t"+sample)
            inputfile = os.path.join(basedir, sample+"_"+s+".root")
            # get sum of weights
            f = ROOT.TFile.Open(inputfile)
            sow[p].append(f.Get("SumOfWeights").GetVal())
            f.Close()
            # get sample dict
            for d in process_dicts :
                if sample in list(d.keys()) :
                    process_dict = d.copy()
                    break
            # get sample
            with uproot.open(inputfile) as f :
                df1.append(ak.to_dataframe(f['events'].arrays(expressions=variables, library='ak')))
                df1[-1]["weight"] = df1[-1]["weight"]/sow[p][-1]*process_dict[sample]['crossSection']*process_dict[sample]['kfactor']*process_dict[sample]['matchingEfficiency']
        df[p][s] = pd.concat(df1, copy=True, ignore_index=True)
        df[p][s]['weight'] = df[p][s]['weight']*process[p]['kfactor']

ttHyy
	nocuts
		mgp8_pp_tth01j_5f_haa
	photons
		mgp8_pp_tth01j_5f_haa
	photons_rel_pt
		mgp8_pp_tth01j_5f_haa
	photons_myy_window_narrow
		mgp8_pp_tth01j_5f_haa
	preselection
		mgp8_pp_tth01j_5f_haa
	lep_channel
		mgp8_pp_tth01j_5f_haa


In [48]:
my_entries = {
    "Selection" : []
}

In [139]:
for p in process.keys() :
    my_entries[process[p]['label']] = []

In [140]:
for s in selection :
    my_entries["Selection"].append(selection[s])
    for p in process.keys() :
        my_entries[process[p]['label']].append(len(df[p][s]))

In [141]:
my_entries = pd.DataFrame(my_entries)
my_entries = my_entries.set_index('Selection')

In [142]:
my_entries

,$ttH$ $\rightarrow$ $\gamma\gamma$ (100 TeV),$tt\gamma\gamma$ (100 TeV),V$\gamma\gamma$+jets (84 TeV)
Selection,,,
All events,3063052,50000,2600000
$\geq$ 2 photons,1786738,22775,687776
Rel. $p_{T}$ cuts,1470172,14912,442476
110 < $m_{\gamma\gamma}$ < 140 GeV,953987,4107,251788
$\geq$ 2 b-jets,586098,2229,17716
$\geq$ 1 lepton,159353,1178,10307


In [53]:
my_yields = {
    "Selection" : []
}

In [54]:
for p in process.keys() :
    my_yields[process[p]['label']] = []

In [55]:
for s in selection :
    my_yields["Selection"].append(selection[s])
    for p in process.keys() :
        my_yields[process[p]['label']].append(df[p][s]["weight"].sum()*lumi)

In [ ]:
my_yields = pd.DataFrame(my_yields)
my_yields = my_yields.set_index('Selection')

In [ ]:
my_yields

In [27]:
my_eff = {
    "Selection" : []
}

In [28]:
for p in process.keys() :
    my_eff[process[p]['label']] = []

In [29]:
for s in selection :
    my_eff["Selection"].append(selection[s])
    for p in process.keys() :
        my_eff[process[p]['label']].append(df[p][s]["weight"].sum()/df[p]["nocuts"]["weight"].sum())

In [30]:
my_eff = pd.DataFrame(my_eff)
my_eff = my_eff.set_index('Selection')

In [31]:
my_eff

,$ttH$ $\rightarrow$ $\gamma\gamma$ (100 TeV),$tt\gamma\gamma$ (100 TeV),V$\gamma\gamma$+jets (84 TeV)
Selection,,,
All events,1.000000,1.000000,1.000000
$\geq$ 2 photons,0.637604,0.501187,0.402119
Rel. $p_{T}$ cuts,0.530038,0.331770,0.259848
110 < $m_{\gamma\gamma}$ < 140 GeV,0.379988,0.096608,0.150743
$\geq$ 2 b-jets,0.224712,0.049331,0.009068
$\geq$ 1 lepton,0.061995,0.025505,0.005290
